<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/SRCNN__3_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [100]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [159]:
import os
import numpy as np
import math
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD, Adam
#from prepare_random_samples import load_h5_data

def create_training_model():
    """
    Eğitim için SRCNN modelini oluşturur.
    Sabit boyutlu giriş şekli (32x32x1) kullanır.
    """
    # SRCNN modeli oluştur
    model = Sequential()

    # İlk evrişim katmanı - Özellik çıkarma
    model.add(Conv2D(
        filters=128,
        kernel_size=(9, 9),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='valid',
        use_bias=True,
        input_shape=(32, 32, 1)
    ))
    model.add(BatchNormalization())
    # İkinci evrişim katmanı - Haritalama
    model.add(Conv2D(
        filters=64,
        kernel_size=(3, 3),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='same',
        use_bias=True
    ))
    model.add(BatchNormalization())

    model.add(Conv2D(
        filters = 64,
        kernel_size = (3,3),
        kernel_initializer = 'glorot_uniform',
        activation = 'relu',
        padding = 'same',
        use_bias = True
    ))
    model.add(BatchNormalization())

    model.add(Conv2D(
        filters = 64,
        kernel_size = (3,3),
        kernel_initializer = 'glorot_uniform',
        activation = 'relu',
        padding = 'same',
        use_bias = True
    ))
    model.add(BatchNormalization())

    # Üçüncü evrişim katmanı - Yeniden yapılandırma
    model.add(Conv2D(
        filters=1,
        kernel_size=(5, 5),
        kernel_initializer='glorot_uniform',
        activation='linear',
        padding='valid',
        use_bias=True
    ))
    model.add(BatchNormalization())

    # Modeli derle
    adam_optimizer = Adam(learning_rate=0.0003)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error']
    )

    return model

In [160]:
def create_prediction_model():
    """
    Tahmin için SRCNN modelini oluşturur.
    Değişken boyutlu giriş şekli (None, None, 1) kullanır.
    """
    # SRCNN modeli oluştur
    model = Sequential()

    # İlk evrişim katmanı - Özellik çıkarma
    model.add(Conv2D(
        filters=128,
        kernel_size=(9, 9),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='valid',
        use_bias=True,
        input_shape=(32, 32, 1)
    ))
    model.add(BatchNormalization())
    # İkinci evrişim katmanı - Haritalama
    model.add(Conv2D(
        filters=64,
        kernel_size=(3, 3),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='same',
        use_bias=True
    ))
    model.add(BatchNormalization())

    model.add(Conv2D(
        filters = 64,
        kernel_size = (3,3),
        kernel_initializer = 'glorot_uniform',
        activation = 'relu',
        padding = 'same',
        use_bias = True
    ))
    model.add(BatchNormalization())

    model.add(Conv2D(
        filters = 64,
        kernel_size = (3,3),
        kernel_initializer = 'glorot_uniform',
        activation = 'relu',
        padding = 'same',
        use_bias = True
    ))
    model.add(BatchNormalization())

    # Üçüncü evrişim katmanı - Yeniden yapılandırma
    model.add(Conv2D(
        filters=1,
        kernel_size=(5, 5),
        kernel_initializer='glorot_uniform',
        activation='linear',
        padding='valid',
        use_bias=True
    ))
    model.add(BatchNormalization())

    # Modeli derle
    adam_optimizer = Adam(learning_rate=0.0003)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error']
    )

    return model


In [161]:
def calculate_psnr(img1, img2):
    """
    İki görüntü arasındaki PSNR'yi (Peak Signal-to-Noise Ratio) hesaplar.
    """
    mse = np.mean((img1 - img2)**2)
    if mse == 0:
        return float('inf')  # MSE 0 ise PSNR sonsuzdur
    max_pixel = 255.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

In [167]:
def train_model():
    """
    SRCNN modelini eğitir ve en iyi modeli kaydeder.
    """
    # Eğitim modelini oluştur
    srcnn_model = create_training_model()
    print(srcnn_model.summary())

    # Eğitim ve doğrulama verilerini yükle
    print("Eğitim verilerini yükleme...")
    train_data, train_labels = load_h5_data("/content/drive/MyDrive/srcnn_dataset/ResNet_SRCNN/train.h5")
    val_data, val_labels = load_h5_data("/content/drive/MyDrive/srcnn_dataset/ResNet_SRCNN/test.h5")

    # Reshape the data to match the expected input shape
    train_data = train_data.transpose(0, 2, 3, 1)  # Transpose to (samples, height, width, channels)
    val_data = val_data.transpose(0, 2, 3, 1)  # Transpose to (samples, height, width, channels)

    # Model kaydetme için callback oluştur
    checkpoint = ModelCheckpoint(
        "SRCNN.h5",
        monitor='val_loss',
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode='min'
    )
    callbacks_list = [checkpoint]

    # Modeli eğit
    print("Model eğitimi başlıyor...")
    srcnn_model.fit(
        train_data, train_labels,
        batch_size=32,
        validation_data=(val_data, val_labels),
        callbacks=callbacks_list,
        shuffle=True,
        epochs=200,
        verbose=1
    )

    print("Eğitim tamamlandı!")

In [165]:
def predict_image(model_path, image_path, output_folder="/content/drive/MyDrive/srcnn_dataset/results"):
    """
    Bir görüntüyü SRCNN ile süper çözünürlüklü hale getirir.

        model_path: Eğitilmiş model ağırlıklarının yolu
        image_path: Girdi görüntüsünün yolu
        output_folder: Sonuçların kaydedileceği klasör
    """
    # Çıktı klasörünü oluştur
    os.makedirs(output_folder, exist_ok=True)

    # Dosya adı bilgilerini hazırla
    base_name = os.path.basename(image_path)
    file_name, _ = os.path.splitext(base_name)
    input_path = os.path.join(output_folder, f"{file_name}_bicubic.png")
    output_path = os.path.join(output_folder, f"{file_name}_srcnn.png")

    # Tahmin modelini oluştur ve ağırlıkları yükle
    srcnn_model = create_prediction_model()
    srcnn_model.load_weights(model_path)
    print(f"Model yüklendi: {model_path}")

    # Orijinal görüntüyü yükle
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # BGR'dan YCrCb'ye dönüştür
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    height, width = img_ycrcb.shape[:2]

    # Y kanalını önce küçült sonra bicubic ile büyüt (düşük çözünürlük simulasyonu)
    y_channel = img_ycrcb[:, :, 0]
    y_channel_lr = cv2.resize(y_channel, (width // 2, height // 2), cv2.INTER_CUBIC)
    y_channel_bicubic = cv2.resize(y_channel_lr, (width, height), cv2.INTER_CUBIC)

    # Bicubic sonucunu kaydet
    img_bicubic = img_ycrcb.copy()
    img_bicubic[:, :, 0] = y_channel_bicubic
    img_bicubic = cv2.cvtColor(img_bicubic, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(input_path, img_bicubic)
    print(f"Bicubic upscaled görüntü kaydedildi: {input_path}")

    # SRCNN için girdiyi hazırla
    input_data = np.zeros((1, height, width, 1), dtype=float)
    input_data[0, :, :, 0] = y_channel_bicubic.astype(float) / 255.0

    # SRCNN tahmini yap
    prediction = srcnn_model.predict(input_data, batch_size=1) * 255.0

    # Değerleri [0, 255] aralığına kırp
    prediction = np.clip(prediction, 0, 255).astype(np.uint8)

    # Tahmini orijinal görüntüye yerleştir (evrişim padding nedeniyle 6 piksel kenarları kırpılır)
    img_srcnn = img_ycrcb.copy()
    img_srcnn[6:-6, 6:-6, 0] = prediction[0, :, :, 0]
    img_srcnn = cv2.cvtColor(img_srcnn, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(output_path, img_srcnn)
    print(f"SRCNN sonucu kaydedildi: {output_path}")

    # PSNR hesaplama
    original_y = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    bicubic_y = cv2.cvtColor(img_bicubic, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    srcnn_y = cv2.cvtColor(img_srcnn, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]

    bicubic_psnr = calculate_psnr(original_y, bicubic_y)
    srcnn_psnr = calculate_psnr(original_y, srcnn_y)

    print(f"Bicubic PSNR: {bicubic_psnr:.2f} dB")
    print(f"SRCNN PSNR: {srcnn_psnr:.2f} dB")
    print(f"PSNR İyileştirmesi: {srcnn_psnr - bicubic_psnr:.2f} dB")


In [168]:
if __name__ == "__main__":
    # Modeli eğit
    train_model()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_410 (Conv2D)                  │ (None, 24, 24, 128)         │          10,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_295              │ (None, 24, 24, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_411 (Conv2D)                  │ (None, 24, 24, 64)          │          73,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_296              │ (None, 24, 24, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_412 (Conv2D)                  │ (None, 24, 24, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_297              │ (None, 24, 24, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_413 (Conv2D)                  │ (None, 24, 24, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_298              │ (None, 24, 24, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_414 (Conv2D)                  │ (None, 20, 20, 1)           │           1,601 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_299              │ (None, 20, 20, 1)           │               4 │
│ (BatchNormalization)                 │                             │                 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 161,029 (629.02 KB)

 Trainable params: 160,387 (626.51 KB)

 Non-trainable params: 642 (2.51 KB)

...

2086/2086 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.6617 - mean_squared_error: 0.6617 - val_loss: 0.1450 - val_mean_squared_error: 0.1450
Epoch 2/200
2081/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0865 - mean_squared_error: 0.0865
Epoch 2: val_loss improved from 0.14496 to 0.00890, saving model to SRCNN.h5


2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0864 - mean_squared_error: 0.0864 - val_loss: 0.0089 - val_mean_squared_error: 0.0089
Epoch 3/200
2074/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0154 - mean_squared_error: 0.0154
Epoch 3: val_loss improved from 0.00890 to 0.00577, saving model to SRCNN.h5


2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0154 - mean_squared_error: 0.0154 - val_loss: 0.0058 - val_mean_squared_error: 0.0058
Epoch 4/200
2073/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0142 - mean_squared_error: 0.0142
Epoch 4: val_loss did not improve from 0.00577
2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0142 - mean_squared_error: 0.0142 - val_loss: 0.0059 - val_mean_squared_error: 0.0059
Epoch 5/200
2076/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0141 - mean_squared_error: 0.0141
Epoch 5: val_loss improved from 0.00577 to 0.00557, saving model to SRCNN.h5


2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0141 - mean_squared_error: 0.0141 - val_loss: 0.0056 - val_mean_squared_error: 0.0056
Epoch 6/200
2077/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0142 - mean_squared_error: 0.0142
Epoch 6: val_loss improved from 0.00557 to 0.00544, saving model to SRCNN.h5


2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0142 - mean_squared_error: 0.0142 - val_loss: 0.0054 - val_mean_squared_error: 0.0054
Epoch 7/200
2083/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0141 - mean_squared_error: 0.0141
Epoch 7: val_loss did not improve from 0.00544
2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0141 - mean_squared_error: 0.0141 - val_loss: 0.0055 - val_mean_squared_error: 0.0055
Epoch 8/200
2086/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0141 - mean_squared_error: 0.0141
Epoch 8: val_loss did not improve from 0.00544
2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0141 - mean_squared_error: 0.0141 - val_loss: 0.0056 - val_mean_squared_error: 0.0056
Epoch 9/200
2074/2086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0140 - mean_squared_error: 0.0140
Epoch 9: val_loss did not improve from 0.00544
2086/2086 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0140 - mean_squared_error: 0.0140 - val_loss: 0.0063 - val_mean_squared_error: 0.0063
Epoch 

In [169]:
predict_image(
        model_path="/content/SRCNN.h5",
        image_path="/content/drive/MyDrive/srcnn_dataset/dataset/test/M0704_img000222.jpg",
        output_folder="/content/drive/MyDrive/srcnn_dataset/results"
    )

Model yüklendi: /content/SRCNN.h5
Bicubic upscaled görüntü kaydedildi: /content/drive/MyDrive/srcnn_dataset/results/M0704_img000222_bicubic.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
SRCNN sonucu kaydedildi: /content/drive/MyDrive/srcnn_dataset/results/M0704_img000222_srcnn.png
Bicubic PSNR: 32.38 dB
SRCNN PSNR: 29.23 dB
PSNR İyileştirmesi: -3.15 dB


### Görüntü iyileşmek yerine daha da kötü oluyor.

*   Nedenini araştır,
*   Veri setini iyileştir.

